In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
def tokenize_word(word):
    return tokenizer.tokenize(word, add_special_tokens=False)

In [2]:
def get_boundaries(tokens, word):
    """Return a set of character positions (1‑based) where a boundary occurs."""
    pos = 0
    boundaries = set()
    for tok in tokens:
        pos += len(tok)
        if pos < len(word):
            boundaries.add(pos)
    return boundaries
gold = []
with open("gold_segmentation.txt", "r", encoding="utf-8") as f:
    for line in f:
        word, seg = line.strip().split("\t")
        parts = seg.split('+')
        pos = 0
        gold_boundaries = set()
        for part in parts:
            pos += len(part)
            if pos < len(word):
                gold_boundaries.add(pos)
        gold.append((word, gold_boundaries))

def evaluate_baseline(tokenizer_func, name):
    total_precision = 0.0
    total_recall = 0.0
    total_f1 = 0.0
    for word, gold_bounds in gold:
        tokens = tokenizer_func(word)
        pred_bounds = get_boundaries(tokens, word)

        intersect = gold_bounds & pred_bounds
        prec = len(intersect) / len(pred_bounds) if pred_bounds else 0.0
        rec = len(intersect) / len(gold_bounds) if gold_bounds else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        total_precision += prec
        total_recall += rec
        total_f1 += f1
    avg_prec = total_precision / len(gold)
    avg_rec = total_recall / len(gold)
    avg_f1 = total_f1 / len(gold)
    print(f"{name}: P={avg_prec:.4f}, R={avg_rec:.4f}, F1={avg_f1:.4f}")
evaluate_baseline(lambda w: tokenizer.tokenize(w, add_special_tokens=False), "IndicBERT")

IndicBERT: P=0.1355, R=0.2633, F1=0.1725


In [5]:
from morfessor import MorfessorIO
io = MorfessorIO()
model = io.read_binary_model_file("model.baseline.bin")
tokens = model.segment(word)[0]
tokens

'सक'

In [6]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor()
sp.load("sp_bpe.model")
tokens = sp.encode_as_pieces(word) 

In [7]:
tokens

['▁सकिन्छ']

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)

trainer = trainers.BpeTrainer(
    vocab_size=16000,
    min_frequency=2,
    special_tokens=["<unk>", "<s>", "</s>", "<pad>"],
)

with open("dataset_ne/ne.txt", "r", encoding="utf-8") as f:
    tokenizer.train_from_iterator(f, trainer=trainer)

tokenizer.save("baseline_byte_bpe.json")
print("Byte-level BPE trained and saved.")

In [7]:
import sentencepiece as spm
from tokenizers import Tokenizer
from morfessor import MorfessorIO

# Load models
tokenizer_byte = Tokenizer.from_file("baseline_byte_bpe.json")

sp_bpe = spm.SentencePieceProcessor()
sp_bpe.Load("sp_bpe.model")

sp_unigram = spm.SentencePieceProcessor()
sp_unigram.Load("sp_unigram.model")

io = MorfessorIO()
model_morf =  io.read_binary_model_file("model.baseline.bin")

# Tokenizer functions
def tok_byte(word):
    return tokenizer_byte.encode(word).tokens

def tok_sp_bpe(word):
    return sp_bpe.EncodeAsPieces(word)

def tok_sp_unigram(word):
    return sp_unigram.EncodeAsPieces(word)

def tok_morfessor(word):
    # Morfessor returns (segmentation, logprob); we take the segmentation list
    return model_morf.segment(word)[0]

# IndicBERT already defined
from transformers import AutoTokenizer
tokenizer_ind = AutoTokenizer.from_pretrained("ai4bharat/indic-bert")
def tok_indic(word):
    return tokenizer_ind.tokenize(word, add_special_tokens=False)

In [8]:
evaluate_baseline(tok_byte, "Byte-level BPE")
evaluate_baseline(tok_sp_bpe, "SentencePiece BPE")
evaluate_baseline(tok_sp_unigram, "SentencePiece Unigram")
evaluate_baseline(tok_morfessor, "Morfessor")
evaluate_baseline(tok_indic, "IndicBERT (pretrained)")

Byte-level BPE: P=0.0604, R=0.0678, F1=0.0614
SentencePiece BPE: P=0.0000, R=0.0000, F1=0.0000
SentencePiece Unigram: P=0.0430, R=0.0427, F1=0.0415
Morfessor: P=0.0823, R=0.3195, F1=0.1259
IndicBERT (pretrained): P=0.1355, R=0.2633, F1=0.1725


In [1]:
from NPBPE_tokenizer import NepBPETokenizer, build_sample_store  # or your custom store

# Load your paradigm store (you might want a larger one than the sample)
paradigm = build_sample_store()  # you'll need to expand this with real Nepali roots

tokenizer_own = NepBPETokenizer(
    paradigm_store=paradigm,
    frozen_strict={"हरू", "हरु"},   # optional
    frozen_ambiguous=set(),
    punctuation_set={"।", "॥", ",", " "}
)

# Read corpus and train
with open("dataset_ne/ne.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

# Normalise the corpus (use your normalize function)
from NPBPE_tokenizer import normalize
corpus_norm = normalize(corpus)

tokenizer_own.train(corpus_norm, vocab_size=8000, latin_budget=0)


: 